# t2i 赛道 · 数据集/题目分析（for 题目构造）

合并自原 `eval_analysis.ipynb`（抽样 + 分布 + 展示）与 `eval_review.ipynb`（题库审阅）：
1. **评测集构建**：委托同目录 `eval_sample.py`（SQL 过滤 + 分层配额抽样；分布/展示两格只读样本清单）
2. **题目审阅全链**：①探针审阅 → ②题库分布 → ③抽样看题 → ④Bagel 生成效果 → ⑤出题人对比 → ⑥判分钻取（批次参数只在顶部 `_BT` 一处）
3. **附录·密度预探（v4 遗留）**：§4b 设计轴存量代理档与四模型横评（只读 + `data/density_probe/` 幂等写）

> 运行：菜单 Run All，或逐 Cell 运行。抽样与题库过滤参数集中在各 cell 顶部，改完重跑该格即可。

In [ ]:
import sys, subprocess, re
import json as _json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, HTML
plt.rcParams['font.family'] = ['Noto Sans CJK SC', 'DejaVu Sans']  # CJK 优先，避免中文豆腐块
plt.rcParams['axes.unicode_minus'] = False


In [ ]:
# ---- 批次版本总开关（切版本只改这一格；探针/题库/Bagel/得分审阅格全从这里取参数） ----
# v5.5 新协议首批（当前；无图出题、恰好 10 条检查、QIB 刻度）：
_BT = dict(
    probe_dir  = 'synth_gen_20260829_v3',             # 探针审阅格：eval_probe generate 输出目录（data/ 下）
    probe_file = 'constraints.jsonl',                 #   约束清单文件名
    samples_f  = 'samples_20260828_v2.jsonl',         # 样本口径清单（data/ 下）
    bank_dir   = 'synth_gen_20260829_v3',             # 题库目录（data/ 下）
    bank_files = ['questions_v55_gpt-5.6-sol.jsonl'], # 题库文件（题库格支持多个）
    bagel_bank = '',                                  # bagel/得分格题库；空 = bank_files[0]
    bagel_run  = 'eval_bagel_v55',                    # bagel 生成批次目录（data/ 下）
)
# 切回 v5.4 pilot 批（取消注释整体替换上面 _BT；v5.4 与 v5.5 不可比，勿混读）：
# _BT = dict(
#     probe_dir='synth_gen_20260828_v2', probe_file='constraints.jsonl',
#     samples_f='samples_20260828_v2.jsonl', bank_dir='synth_gen_20260828_v2',
#     bank_files=['questions_v54_gpt-5.6-sol.jsonl'],
#     bagel_bank='', bagel_run='eval_bagel_v54')
print('当前批次 →', _BT['bank_dir'], '/', _BT['bagel_run'])


## 评测集构建（eval_sample 分层抽样）

抽样直接委托同目录 `eval_sample.py`（机制与全部参数见脚本头 docstring 与 `--help`）：**每次运行全量重抽**，覆盖写 `data/samples.jsonl` 并清除 `data/images/` 旧样本拷贝（样本分布跟随最新数据）。

In [ ]:
# ---- 评测抽样：委托同目录 eval_sample.py ----
# 默认幂等：仅当 SAMPLES 不存在时才抽样；如需强制重抽改 RESAMPLE=True
RESAMPLE        = False     # ← True=强制重抽（覆盖写 SAMPLES）；False=缺失时才抽（推荐）
EVAL_N          = 10        # ← 抽样张数（仅 RESAMPLE=True 时生效）
EVAL_FILTER     = "quality >= 9.5 and width >= 1080 and height >= 1080 and height <=1600 and width <= 1600"
                              # ← 候选过滤条件（duckdb SQL WHERE）
EVAL_PER_INST   = 1         # ← 每实例限张数
EVAL_SEED       = 20260826  # ← 抽样种子（固定 → 可复现；与历史 pilot 必须用同种子）
EVAL_DRY_RUN    = False     # ← True=只打印配额分配不落盘
EVAL_OUT        = None      # ← 样本清单输出路径；None=脚本默认（data/samples.jsonl）
EVAL_IMG_DIR    = None      # ← 图片拷贝目录；None=脚本默认（data/images）

SAMPLES = Path(EVAL_OUT) if EVAL_OUT else Path('data/samples.jsonl')

if RESAMPLE or not SAMPLES.exists():
    cmd = [sys.executable, str(Path('eval_sample.py').resolve()),
           '--n', str(EVAL_N), '--filter', EVAL_FILTER,
           '--per-instance', str(EVAL_PER_INST), '--seed', str(EVAL_SEED)]
    if EVAL_OUT:
        cmd += ['--out', str(EVAL_OUT)]
    if EVAL_IMG_DIR:
        cmd += ['--img-dir', str(EVAL_IMG_DIR)]
    if EVAL_DRY_RUN:
        cmd.append('--dry-run')
    print(f'>> RESAMPLE={RESAMPLE or not SAMPLES.exists()}: 跑 eval_sample.py')
    assert subprocess.run(cmd).returncode == 0, 'eval_sample.py 异常退出'
else:
    print(f'>> RESAMPLE=False 且 {SAMPLES} 已存在：跳过抽样（幂等模式）')

rows = [_json.loads(l) for l in open(SAMPLES, encoding='utf-8') if l.strip()]
sdf = pd.DataFrame(rows)
EVAL_DIR = SAMPLES.parent                        # 图片按清单内相对 image 字段 + 清单所在目录解析
print(f'评测集: {len(sdf):,} 样本 / {sdf.instance.nunique():,} 实例 → {SAMPLES.resolve()}')
with pd.option_context('display.max_colwidth', 60):
    display(sdf[['sample_id', 'instance', 'l1', 'l2', 'quality', 'focus', 'kb_match']].head(20))


In [ ]:
# ---- 评测集分布分析（只读 samples.jsonl；复用上一格 sdf）----
per_inst = sdf.instance.value_counts()
print(f'总图数: {len(sdf):,}   去重实例: {sdf.instance.nunique():,}   覆盖 (L1,L2) 分支: {sdf.groupby(["l1","l2"]).ngroups}')
print('每实例图数分布: ' + '   '.join(f'{k} 张: {v:,} 实例' for k, v in per_inst.value_counts().sort_index().items()))

fig, axes = plt.subplots(2, 3, figsize=(18, 9))
# ① L1 分支分布
l1 = sdf.l1.value_counts()
axes[0][0].barh(l1.index[::-1], l1.values[::-1], color='#8ab')
for i, v in enumerate(l1.values[::-1]):
    axes[0][0].text(v, i, f' {v:,}', va='center', fontsize=8)
axes[0][0].set_title(f'L1 分支分布（{len(l1)} 个）')
# ② (L1,L2) 分支 Top20
br = sdf.groupby(['l1', 'l2']).size().sort_values().tail(20)
axes[0][1].barh([' / '.join(ix) for ix in br.index], br.values, color='#7a9')
axes[0][1].tick_params(axis='y', labelsize=7)
axes[0][1].set_title(f'(L1,L2) 分支样本量 Top20（共 {sdf.groupby(["l1","l2"]).ngroups} 分支）')
# ③~⑤ 打分字段分布
for ax, f in [(axes[0][2], 'quality'), (axes[1][0], 'focus'), (axes[1][1], 'kb_match')]:
    vals = pd.to_numeric(sdf[f], errors='coerce').dropna()
    lo, hi = float(vals.min()), float(vals.max())
    ax.hist(vals, bins=(np.linspace(lo, hi, 21) if hi > lo else 1), edgecolor='white', alpha=0.85)
    ax.axvline(vals.mean(), color='red', ls='--', lw=1, label=f'mean={vals.mean():.2f}')
    ax.legend(fontsize=8)
    ax.set_title(f'{f}  min={lo:g}  max={hi:g}  null={len(sdf)-len(vals)}')
# ⑥ 短边分布
se = np.minimum(sdf.width, sdf.height).dropna()
axes[1][2].hist(se, bins=40, edgecolor='white', color='#97a', alpha=0.85)
axes[1][2].legend(fontsize=8)
axes[1][2].set_title(f'图片短边分布（中位 {se.median():.0f}px）')
plt.tight_layout(); plt.show()

In [ ]:
# ---- 评测集抽样展示（图卡；复用 sdf / EVAL_DIR）----
N_CASES = 20     # ← 改这里：抽几张展示

show = sdf.sample(min(N_CASES, len(sdf)), random_state=42)
for _, r in show.iterrows():
    ip = EVAL_DIR / r['image']
    img = (f'<img src="{ip}" loading="lazy" style="max-height:280px;max-width:380px;object-fit:contain">'
           if ip.is_file() else '<div style="color:#c00">图缺失</div>')
    kv = ''.join(f'<div style="font-size:12px;margin:1px 0"><b>{k}</b>: {v}</div>' for k, v in [
        ('sample', f'{r["sample_id"]} · {r["instance"]}'),
        ('branch', f'{r["l1"]} / {r["l2"]}'),
        ('metrics', f'quality={r["quality"]}  focus={r["focus"]}  kb_match={r["kb_match"]}  {r["width"]}×{r["height"]}'),
        ('caption', str(r.get('caption', ''))[:220]),
    ])
    display(HTML(f'<div style="display:flex;gap:12px;border:1px solid #ddd;padding:8px;margin:8px 0;align-items:flex-start">'
                 f'{img}<div style="min-width:0">{kv}</div></div>'))

## 题目审阅全链（探针 → 题库 → Bagel → 判分；for 题目构造）

六格顺链读下来；批次参数全部来自顶部 `_BT` 总开关，各格只留查看过滤项（如 VIEW_INST / VIEW_SAMPLE / F_JUDGE）：
1. **探针审阅**：五路原文 × sol 合并清单 × 样本图对读（`_BT['probe_dir']`；v5.5 合并册 10~12 条、QIB 刻度）
2. **题库加载 + 分布分析**：题源 `_BT['bank_files']`（本格初始化 `T2I_ROOT`/题源字典，后续格复用；须先跑）
3. **抽样看题**：样本图 + 元数据卡片 + 各出题人完整原题（不截断）
4. **Bagel 生成效果**：`_BT['bagel_run']` 生成图 × 题面并排
5. **出题人质量对比**：样本 × 出题人 × 判官得分矩阵（`JUDGES=[]` 自动发现该批判官）
6. **判分明细钻取**：逐条 check 得分与判官理由

> 机审说明：v5.5 结构机审随 `eval_synthesize.py` 出题实时执行（唯一机审，只告警不拦截、无独立产物），验收 = 出题日志 0 warn，故无独立质检格。


In [ ]:
# ---- ① 探针审阅（五路原文 × sol 合并清单 × 样本图；eval_probe.py generate 产物）----
import html as _html
import json as _json
from pathlib import Path
from IPython.display import display, HTML

# ====== 参数 ======
assert '_BT' in globals(), '先跑顶部「批次版本总开关」格'
PROBE_DIR   = _BT['probe_dir']             # ← 总开关
PROBE_FILE  = _BT.get('probe_file') or 'constraints.jsonl'
SAMPLES_F   = _BT['samples_f']             # ← 总开关
VIEW_INST   = ''    # 只看某实例（如 '鱼雷发射管'）；空 = 全部
SHOW_RAW    = True  # 是否展开五路原始约束清单（raw/ 逐模型）
# ==================

try:
    T2I_ROOT
except NameError:
    T2I_ROOT = Path.cwd() if (Path.cwd() / 'data' / 'samples.jsonl').exists() else Path.cwd() / 'benchmark' / 't2i'
_EVAL_DIR = T2I_ROOT / 'data'
_pd = _EVAL_DIR / PROBE_DIR
_cons_f = _pd / PROBE_FILE
assert _cons_f.exists(), f'{_cons_f} 不存在，先跑 eval_probe.py generate'

recs = [_json.loads(l) for l in _cons_f.open(encoding='utf-8') if l.strip()]
samp = {_json.loads(l)['instance']: _json.loads(l)
        for l in (_EVAL_DIR / SAMPLES_F).open(encoding='utf-8') if l.strip()}

def _e(x): return _html.escape(str(x))

def _raw_items(name):
    """raw/ 里逐模型原始约束（解析 API 响应 content 中的 constraints）。"""
    out = []
    for p in sorted((_pd / 'raw').glob(f'{name}_*.json')):
        if p.name.endswith('_merge.json'):
            continue
        model = p.stem[len(name) + 1:].replace('_', '/')
        try:
            msg = _json.loads(p.read_text(encoding='utf-8'))['choices'][0]['message']
            content = msg.get('content') or msg.get('reasoning_content') or ''
            m = re.search(r'\{.*\}', content, re.S)
            items = _json.loads(m.group(0)).get('constraints') or [] if m else []
        except Exception:
            items = []
        out.append((model, items))
    return out

import re
n_pass = sum(1 for r in recs if r.get('_threshold_pass'))
allc = [c for r in recs for c in r.get('constraints', [])]
neg = sum(1 for c in allc if c['polarity'] == 'must_not_have')
var = sum(1 for c in allc if c.get('variants'))
n_pre   = sum(1 for r in recs if r.get('_precheck_done'))
n_prerm = sum(len(r.get('_precheck_removed') or []) for r in recs)
print(f"批次 {PROBE_DIR} | 实例 {len(recs)} | 通过阈值 {n_pass} ({n_pass/max(1,len(recs))*100:.0f}%) | "
      f"约束 {len(allc)} 条（{len(allc)/max(1,len(recs)):.1f}/实例）| "
      f"负向 {neg/max(1,len(allc))*100:.0f}% | 带变体 {var/max(1,len(allc))*100:.0f}% | "
      f"预检 {n_pre}/{len(recs)}、剔除 {n_prerm} 条")

for r in recs:
    name = r['instance']
    if VIEW_INST and name != VIEW_INST:
        continue
    s = samp.get(name, {})
    ip = _EVAL_DIR / s.get('image', '') if s else None
    img = (f'<img src="{ip}" style="max-height:300px;max-width:400px;object-fit:contain">'
           if ip and ip.is_file() else '<div style="color:#c00">图缺失</div>')
    badge = '#2a7a4b' if r.get('_threshold_pass') else '#a33333'
    pre_t = ('<span style="color:#2a7a4b;font-size:12px">✓预检</span>' if r.get('_precheck_done')
             else '<span style="color:#a07a33;font-size:12px">未预检</span>')
    head = (f'<div style="font-size:15px;font-weight:700;margin:8px 0 2px">'
            f'<span style="color:{badge}">{"PASS" if r.get("_threshold_pass") else "FAIL"}</span> '
            f'{pre_t} {_e(name)} <span style="font-size:12px;color:#666">{_e(" / ".join(s.get("mount_paths") or []))}</span></div>')
    rows = []
    for c in r.get('constraints', []):
        v = f' <span style="color:#25648a">仅子变体: {_e(c["scope"])}</span>' if c.get('scope') else ''
        pol = ('<b style="color:#a33333">NEG</b> ' if c['polarity'] == 'must_not_have' else '')
        rows.append(f'<div style="margin:3px 0">[{c.get("id")}] {pol}{_e(c["constraint"])}{v}'
                    f'<span style="color:#999;font-size:11px"> ／ {c.get("tier","")} · src={c.get("source","")}</span></div>')
    rm = ''.join(f'<div style="margin:2px 0;color:#888">✗ {_e(x.get("constraint",""))} <i>({_e(x.get("reason",""))})</i></div>'
                 for x in r.get('removed', []))
    prm = ''.join(f'<div style="margin:2px 0;color:#a07a33">✗ {_e(x.get("constraint",""))} <i>({_e(x.get("reason",""))})</i></div>'
                  for x in r.get('_precheck_removed') or [])
    note = f'<div style="color:#777;font-size:12px;margin-top:4px">notes: {_e(r.get("notes",""))}</div>' if r.get('notes') else ''
    raw_html = ''
    if SHOW_RAW:
        segs = []
        for model, items in _raw_items(name):
            li = ''.join(f'<li>{_e(c.get("constraint",""))}'
                         + (f' <span style="color:#25648a">[变体: {"、".join(map(str,c.get("variants") or []))}]</span>' if c.get('variants') else '')
                         + (' <b style="color:#a33333">[NEG]</b>' if c.get('polarity') == 'must_not_have' else '') + '</li>'
                         for c in items)
            segs.append(f'<div style="margin:4px 0"><b style="color:#555">{_e(model)}</b> ({len(items)} 条)'
                        + (f'<ul style="margin:2px 0 2px 18px;padding:0">{li}</ul>' if items else ' <i style="color:#c00">空</i>') + '</div>')
        raw_html = ('<div style="margin-top:8px;font-size:13px"><b>五路原文</b>'
                    + ''.join(segs) + '</div>')
    gm = ' '.join(f'<span style="color:{"#2a7a4b" if m.get("parse_ok") and m.get("n") else "#c00"}">{_e(m["model"].split("/")[-1])}:{m.get("n")}</span>'
                  for m in r.get('_gen_models', []))
    html = (head + f'<div style="font-size:12px;color:#666">{gm} | 原始 {r.get("_n_total_items","?")} 条 → 合并 {len(r.get("constraints",[]))} 条</div>'
            + '<table><tr><td style="vertical-align:top;padding-right:12px">' + img + '</td>'
            + '<td style="vertical-align:top">' + ''.join(rows)
            + (f'<div style="margin-top:6px;font-size:12px"><b>sol 合并剔除 {len(r.get("removed",[]))} 条</b>{rm}</div>' if rm else '')
            + (f'<div style="margin-top:4px;font-size:12px"><b>预检剔除 {len(r.get("_precheck_removed") or [])} 条</b>{prm}</div>' if prm else '')
            + note + '</td></tr></table>' + raw_html
            + '<hr style="border:none;border-top:1px dashed #ccc;margin:14px 0">')
    display(HTML(html))


In [ ]:
# ---- ② 题库加载 + 分布分析（题源=_BT；本格初始化 T2I_ROOT/题源字典，后续格复用）----
import json as _json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

assert '_BT' in globals(), '先跑顶部「批次版本总开关」格'
SYNTH_DIR_NAME = _BT['bank_dir']           # ← 总开关
SAMPLES_FILE   = _BT['samples_f']          # ← 总开关
QUESTION_SOURCE = _BT['bank_files']        # ← 总开关
                                           # ← 最新批题源（data/{SYNTH_DIR_NAME}/ 下文件名列表）；
                                           #   v5.4 批示例：SYNTH_DIR_NAME='synth_gen_20260828_v2',
                                           #   SAMPLES_FILE='samples_20260828_v2.jsonl',
                                           #   QUESTION_SOURCE=['questions_v54_gpt-5.6-sol.jsonl']
# cwd 自适应：notebook 在 benchmark/t2i/ 跑或从仓库根跑都能找到数据
T2I_ROOT = Path.cwd() if (Path.cwd() / 'data' / 'samples.jsonl').exists() else Path.cwd() / 'benchmark' / 't2i'
assert (T2I_ROOT / 'data' / 'samples.jsonl').exists(), f'找不到 samples.jsonl（cwd={Path.cwd()}）'
SYNTH_DIR = T2I_ROOT / 'data' / SYNTH_DIR_NAME
try:
    _EVAL_DIR = EVAL_DIR          # 评测抽样格已跑则复用
except NameError:
    _EVAL_DIR = T2I_ROOT / 'data'

# 「这次抽样」口径：当前 samples.jsonl 权威清单的 image 字段（评测抽样格产物）
_samples_fp = _EVAL_DIR / SAMPLES_FILE
assert _samples_fp.exists(), f'样本清单不存在: {_samples_fp}'
sample_meta = {r['image']: r for r in map(_json.loads, _samples_fp.open(encoding='utf-8'))}

qs = []
for name in QUESTION_SOURCE:
    fp = SYNTH_DIR / name
    assert fp.exists(), f'题源不存在: {fp}（可改 QUESTION_SOURCE）'
    for line in fp.read_text(encoding='utf-8').splitlines():
        if line.strip():
            qs.append(_json.loads(line))
_matched = [q for q in qs if q.get('_sample_image') in sample_meta]
qs = _matched
V53 = any('fidelity_checks' in q for q in qs)   # schema 探测：v5.3 有保真检查四分组
print(f'题库: {len(qs)} 题 / {len({q["_sample_image"] for q in qs})} 样本 / '
      f'{len({q["_generator_model"] for q in qs})} 模型 | schema={"v5.3" if V53 else "v5.2 及更早"} ← {", ".join(QUESTION_SOURCE)}')

# ====== 过滤参数（改完重跑本 cell；空 = 不过滤）======
QIDS           = []     # 精确 qid
SAMPLE_KEYS    = []     # 样本号前缀或 query 标签，如 ['0001', '杜波依斯']
MODELS         = []     # 生成模型，如 ['openrouter/z-ai/glm-5.3-flash']
INVOCATIONS    = []     # v5.3 调用方式 'L2'/'L3'（旧批兼容：也匹配 difficulty）
KNOWLEDGE_DIMS = []     # 知识维度
FACET_TAGS     = []     # 须同时命中的判分项（v5.3=对齐/质量/美感三组词表；旧批=facet_tags）
# ====== 分布维度 GROUP_BY（≤6 个，画图 2×3）======
GROUP_BY = (['_generator_model', 'invocation', 'knowledge_dim',
             ('invocation', '_generator_model'), 'facets', 'sample'] if V53 else
            ['_generator_model', 'difficulty', 'knowledge_dim',
             ('difficulty', '_generator_model'), 'facet_tags', 'sample'])
# ====================================================

def _sid(q):
    si = q.get('_sample_image') or ''
    return si.split('/')[1][:4] if si.startswith('images/') else si

def _all_facets(q):
    if 'fidelity_checks' in q:
        return ([c.get('facet') for c in (q.get('alignment_checks') or [])]
                + list(q.get('quality_facets') or []) + list(q.get('aesthetic_facets') or []))
    return q.get('facet_tags') or []

def _keep(q):
    if QIDS and q.get('qid') not in QIDS:
        return False
    if SAMPLE_KEYS and not (_sid(q) in SAMPLE_KEYS or q.get('_query_label') in SAMPLE_KEYS):
        return False
    if MODELS and q.get('_generator_model') not in MODELS:
        return False
    if INVOCATIONS and (q.get('invocation') or q.get('difficulty')) not in INVOCATIONS:
        return False
    if KNOWLEDGE_DIMS and q.get('knowledge_dim') not in KNOWLEDGE_DIMS:
        return False
    if FACET_TAGS and not set(FACET_TAGS) <= set(_all_facets(q)):
        return False
    return True

def _dim_values(dim, items):
    if dim == 'facets':
        return [t for q in items for t in _all_facets(q)]
    if dim == 'facet_tags':
        return [t for q in items for t in (q.get('facet_tags') or [])]
    if dim == 'sample':
        return [f'{_sid(q)} {q.get("_query_label", "")}' for q in items]
    if dim == 'invocation':
        return [q.get('invocation') or q.get('difficulty') or '?' for q in items]
    return [q.get(dim, '?') for q in items]

if not qs:
    print('题库为空（先跑 eval_synthesize.py 出题）')
else:
    sel = [q for q in qs if _keep(q)]
    print(f'过滤命中 {len(sel)} 题 / {len({_sid(q) for q in sel})} 样本 / {len({q["_generator_model"] for q in sel})} 模型')

    fig, axes = plt.subplots(2, 3, figsize=(18, 9))
    for ax, g in zip(axes.flat, GROUP_BY):
        if isinstance(g, tuple):
            a, b = g
            ct = pd.crosstab(pd.Series(_dim_values(a, sel)), pd.Series(_dim_values(b, sel)))
            ct.plot.barh(ax=ax, stacked=True, width=0.7)
            ax.invert_yaxis()
            ax.tick_params(axis='y', labelsize=7)
            ax.legend(fontsize=7)
            ax.set_title(f'{a} × {b} 堆叠分布（{len(sel)} 题）')
        else:
            s = pd.Series(_dim_values(g, sel)).value_counts()
            ax.barh(s.index[::-1], s.values[::-1], color='#8ab')
            for i, v in enumerate(s.values[::-1]):
                ax.text(v, i, f' {v}', va='center', fontsize=8)
            ax.tick_params(axis='y', labelsize=7)
            ax.set_title(f'{g} 分布（{len(s)} 种 / {s.sum()} 条）')
    for ax in axes.flat[len(GROUP_BY):]:
        ax.axis('off')
    plt.tight_layout(); plt.show()

    # ---- 命中题目明细（全量字段在 qdf）----
    _COL_ORDER = (['qid', '_generator_model', '_job_sample', '_query_label', 'invocation',
                   'knowledge_dim', 'gen_prompt', 'fidelity_checks', 'alignment_checks',
                   'quality_facets', 'aesthetic_facets', 'leak_check', 'notes', '_sample_image']
                  if V53 else
                  ['qid', '_generator_model', 'sample_id', '_query_label', 'difficulty',
                   'knowledge_dim', 'suite', 'facet_tags', 'gen_prompt', 'gate_spec',
                   'implicit_checks', 'evidence_audit', 'expected_failure_modes', 'notes',
                   '_sample_image', 'task'])
    cols = [c for c in _COL_ORDER if any(c in q for q in sel)]
    cols += sorted({k for q in sel for k in q} - set(cols))

    def _flat(v):
        if isinstance(v, dict):
            return _json.dumps(v, ensure_ascii=False)
        if isinstance(v, list) and all(isinstance(x, str) for x in v):
            return '、'.join(v)
        if isinstance(v, list):
            return _json.dumps(v, ensure_ascii=False)
        return v

    qdf = pd.DataFrame([{k: _flat(q.get(k)) for k in cols} for q in sel])
    print(f'命中题目明细（{len(qdf)} 题 × {len(cols)} 字段；qdf 存全量字段值）:')
    with pd.option_context('display.max_colwidth', 42, 'display.max_rows', None):
        display(qdf)


In [ ]:
# ---- ③ 抽样看题（HTML 图卡：样本图 + 元数据 + 各出题人原题完整输出，不截断）----
import re, random as _rnd, html as _html
import json as _json
from pathlib import Path
from IPython.display import display, HTML

# ====== 参数 ======
VIEW_SAMPLE = ''       # 指定样本：样本号前缀（如 '0009'）或 query 标签（如 '广东醒狮'）；空 = 随机抽
VIEW_N      = 5        # 随机抽几个样本（VIEW_SAMPLE 非空时忽略）
SEED        = 7        # 随机种子
# ==================

try:
    T2I_ROOT                                    # 加载格已跑
    _EVAL_DIR = T2I_ROOT / 'data'
except NameError:
    T2I_ROOT = Path.cwd() if (Path.cwd() / 'data' / 'samples.jsonl').exists() else Path.cwd() / 'benchmark' / 't2i'
    _EVAL_DIR = T2I_ROOT / 'data'
REPO_ROOT = T2I_ROOT.parent.parent              # benchmark/t2i → 仓库根
BLOBS_DIR = REPO_ROOT / 'datasets' / 'demiwtg' / 'blobs'
DIFF_COLOR = {'L1': '#2a7a4b', 'L2': '#b07020', 'L3': '#a33333'}

def _esc(x):
    return _html.escape(str(x))

def _sid(q):
    si = q.get('_sample_image') or ''
    return si.split('/')[1][:4] if si.startswith('images/') else si

def _resolve_image(img_rel):
    """题库 _sample_image → 图片路径；EVAL_DIR 优先，缺失回退 blobs 内容寻址原图"""
    p = _EVAL_DIR / img_rel
    if p.exists():
        return p
    m = re.search(r'_([a-f0-9]{8,})\.[a-z]+$', img_rel)
    if m:
        d = BLOBS_DIR / m.group(1)[:2]
        if d.exists():
            hits = list(d.glob(f'{m.group(1)}*'))
            if hits:
                return hits[0]
    return None

def _section(title, body):
    return f'<div style="margin-top:6px;font-size:12px"><b>[{title}]</b>{body}</div>'

def _rubric_html(rub):
    if not rub:
        return ''
    keys = ('tier_0', 'tier_1', 'tier_2') if any(k in rub for k in ('tier_0', 'tier_1', 'tier_2')) else ('0', '1', '2')
    names = {'tier_0': '0', 'tier_1': '及格', 'tier_2': '满分', '0': '0', '1': '及格', '2': '满分'}
    seg = '　'.join(f'<b>{names[s]}档</b> {_esc(rub[s])}' for s in keys if rub.get(s))
    return f'<div style="margin-left:14px">{seg}</div>' if seg else ''

def _checks_html(checks):
    out = []
    for i, c in enumerate(checks or [], 1):
        if not isinstance(c, dict):
            out.append(f'<div style="margin:5px 0">{i}. {_esc(c)}</div>')
            continue
        w = c.get('weight')
        ws = f'[{w:g}] ' if w is not None else ''
        head = f'<b>{i}. {ws}{_esc(c.get("check") or c.get("check_text") or c.get("point") or "")}</b>'
        rows = [head]
        if c.get('knowledge'):
            rows.append(f'<div style="color:#555;margin-left:14px">知识: {_esc(c["knowledge"])}</div>')
        rows.append(_rubric_html(c.get('rubric')))
        if c.get('acceptable_variants'):
            av = '；'.join(str(x) for x in c['acceptable_variants'])
            rows.append(f'<div style="color:#666;margin-left:14px">允许变体: {_esc(av)}</div>')
        out.append(f'<div style="margin:5px 0">{"".join(rows)}</div>')
    return ''.join(out)

def _fidelity_html(checks):
    out = []
    for i, c in enumerate(checks or [], 1):
        an = c.get('anchor') or {}
        refs = '、'.join(map(str, an.get('constraint_refs') or [])) or '—'
        badge = {'direct': '#2a7a4b', 'combination': '#25648a', 'derivation': '#a33333'}.get(an.get('type'), '#666')
        head = (f'<b>{i}. {_esc(c.get("check", ""))}</b> '
                f'<span style="background:{badge};color:#fff;border-radius:3px;padding:0 4px;font-size:11px">'
                f'{_esc(an.get("type", "?"))} ← [{refs}]</span>')
        rows = [head, _rubric_html(c.get('rubric'))]
        if an.get('derivation_chain'):
            rows.append(f'<div style="color:#a33333;margin-left:14px">推导链: {_esc(an["derivation_chain"])}</div>')
        if c.get('variants_used'):
            rows.append(f'<div style="color:#666;margin-left:14px">变体: {_esc(c["variants_used"])}</div>')
        if c.get('visibility_requirement'):
            rows.append(f'<div style="color:#888;margin-left:14px">可见条件: {_esc(c["visibility_requirement"])}</div>')
        out.append(f'<div style="margin:6px 0">{"".join(r for r in rows if r)}</div>')
    return ''.join(out)

def _q_card(q):
    col = DIFF_COLOR.get(q.get('invocation') or q.get('difficulty'), '#666')
    inv = q.get('invocation') or q.get('difficulty') or '?'
    parts = [
        f'<div style="font-size:13px"><b>{_esc(q.get("qid", "?"))}</b> · {_esc(q.get("_generator_model", "?"))} · '
        f'<span style="color:{col};font-weight:bold">{_esc(inv)}</span> · '
        f'{_esc(q.get("knowledge_dim", "?"))}</div>',
        f'<div style="padding:6px;margin:6px 0;font-size:13px">{_esc(q.get("gen_prompt") or q.get("stem") or "")}</div>',
    ]
    if 'fidelity_checks' in q:   # v5.3
        parts.append(f'<div style="margin-top:6px;font-size:12px"><b>[真实保真度]</b>（{len(q.get("fidelity_checks") or [])} 条，唯一知识源=已核硬约束清单）{_fidelity_html(q.get("fidelity_checks"))}</div>')
        ac = q.get('alignment_checks') or []
        if ac:
            body = []
            for i, c in enumerate(ac, 1):
                body.append(f'<div style="margin:6px 0"><b>{i}. {_esc(c.get("check", ""))}</b> '
                            f'<span style="background:#7a5ca8;color:#fff;border-radius:3px;padding:0 4px;font-size:11px">{_esc(c.get("facet", "?"))}</span>'
                            + _rubric_html(c.get('rubric')) + '</div>')
            parts.append(f'<div style="margin-top:6px;font-size:12px"><b>[文本一致性]</b>（{len(ac)} 条）{"".join(body)}</div>')
        qf, af = q.get('quality_facets') or [], q.get('aesthetic_facets') or []
        parts.append(_section('质量', ' ' + '、'.join(map(_esc, qf)) + '　<b>美感</b> ' + '、'.join(map(_esc, af))))
        lk = q.get('leak_check') or {}
        tbl = lk.get('对照表') or []
        n_leak = sum(1 for r in tbl if str(r.get('题面中是否出现', '无')).strip() != '无')
        lk_col = '#2a7a4b' if n_leak == 0 else '#c0392b'
        parts.append(_section('防泄漏', f' <span style="color:{lk_col};font-weight:bold">{_esc(lk.get("结论", "?"))}</span>（对照 {len(tbl)} 条，泄漏 {n_leak}）'))
    else:                        # v5.2 及更早
        gs = q.get('gate_spec') or {}
        if gs:
            parts.append(_section('gate_spec', f' required_subjects: {_esc(gs.get("required_subjects", []))} · 主题: {_esc(gs.get("theme_definition", ""))}'))
        if q.get('facet_tags'):
            parts.append(_section('facet_tags', ' ' + '、'.join(map(_esc, q['facet_tags']))))
        if q.get('implicit_checks'):
            parts.append(f'<div style="margin-top:6px;font-size:12px"><b>[implicit_checks]</b>{_checks_html(q["implicit_checks"])}</div>')
        ea = q.get('evidence_audit') or {}
        if ea:
            rows = []
            for k in ('visible_facts', 'anchor_features'):
                if ea.get(k):
                    rows.append(f'<div style="color:#555">{k}: ' + '；'.join(map(_esc, ea[k])) + '</div>')
            tc = ea.get('trigger_check') or ea.get('trigger') or ''
            if tc:
                rows.append(f'<div>trigger_check: {_esc(tc)}</div>')
            if ea.get('counterexample_test'):
                rows.append(f'<div>counterexample_test: {_esc(ea["counterexample_test"])}</div>')
            parts.append(_section('evidence_audit', ''.join(rows)))
        if q.get('expected_failure_modes'):
            fm = ''.join(f'<div>· {_esc(m)}</div>' for m in q['expected_failure_modes'])
            parts.append(_section('expected_failure_modes', f'<div>{fm}</div>'))
    if q.get('notes'):
        parts.append(_section('notes', f' {_esc(q["notes"])}'))
    return (f'<div style="border:1px solid #ddd;border-left:4px solid {col};padding:8px 12px;margin:8px 0">'
            + ''.join(parts) + '</div>')

def _sample_card(img_rel, grp):
    m = sample_meta.get(img_rel) or {}
    sid = _sid(grp[0]) if grp else img_rel
    label = grp[0].get('_query_label', '') if grp else ''
    ip = _resolve_image(img_rel)
    img = (f'<img src="{ip}" loading="lazy" style="max-height:300px;max-width:400px;object-fit:contain">'
           if ip else '<div style="color:#c00">图缺失</div>')
    kv = ''.join(f'<div style="font-size:12px;margin:1px 0"><b>{k}</b>: {v}</div>' for k, v in [
        ('样本', f'{sid} · {label} · {len(grp)} 题'),
        ('模型', '、'.join(_esc(q.get('_generator_model', '?')) for q in grp)),
        ('branch', f'{m.get("l1", "?")} / {m.get("l2", "?")}'),
        ('metrics', f'quality={m.get("quality", "?")} focus={m.get("focus", "?")} kb_match={m.get("kb_match", "?")}'),
        ('caption', _esc(m.get('caption', ''))),
    ] if v)
    display(HTML(f'<div style="display:flex;gap:12px;border:1px solid #999;'
                 f'padding:8px;margin:14px 0 2px;align-items:flex-start">{img}<div style="min-width:0">{kv}</div></div>'))

if not qs:
    print('题库为空（先跑上一格加载）')
else:
    if VIEW_SAMPLE:
        pool = sorted({q['_sample_image'] for q in qs if q.get('_sample_image')
                       and (_sid(q).startswith(VIEW_SAMPLE) or (q.get('_query_label') or '') == VIEW_SAMPLE)})
    else:
        all_imgs = sorted({q['_sample_image'] for q in qs if q.get('_sample_image')})
        pool = sorted(_rnd.Random(SEED).sample(all_imgs, min(VIEW_N, len(all_imgs))))
    print(f'待查看样本: {len(pool)} 个（{[p.split("/")[1][:4] for p in pool]}）')
    if not pool:
        print(f'!! 无匹配样本（VIEW_SAMPLE={VIEW_SAMPLE!r}）')
    for img_rel in pool:
        grp = sorted((q for q in qs if q.get('_sample_image') == img_rel), key=lambda x: x.get('_generator_model', ''))
        _sample_card(img_rel, grp)
        for q in grp:
            display(HTML(_q_card(q)))


In [ ]:
# ---- ④ Bagel 生成效果查看 + 原图基线判分（样本图 × 题面 + 生成图并排；原图直接作答案判分 → ⑤⑥ 消费）----
import json as _json, html as _html, subprocess as _subp, sys as _sys
from pathlib import Path
from IPython.display import display, HTML

# ====== 参数 ======
assert '_BT' in globals(), '先跑顶部「批次版本总开关」格'
GEN_RUN      = _BT['bagel_run']            # ← 总开关
GEN_BANK_DIR = _BT['bank_dir']             # ← 总开关
GEN_BANK     = _BT['bagel_bank'] or _BT['bank_files'][0]   # ← 总开关
GEN_SAMPLE  = ''                          # 指定样本号（如 '0009'）；空 = 全部样本
GEN_ORIG    = True                        # 原图基线判分：样本原图直接作答案，同套 checks 同判官（按 qid 断点续跑）
GEN_ORIG_JUDGES   = []                    # 判官名单；空 = RUN_DIR 已有 scores_<judge>.jsonl 全部（与生成图同判官方可比）
GEN_ORIG_MODEL    = ''                    # 判官完整路由（如 openrouter/google/gemini-3.7-flash）；空 = 网关按判官名反查
GEN_ORIG_ENDPOINT = 'http://127.0.0.1:4001/v1/chat/completions'
GEN_ORIG_WORKERS  = 2
# ==================

try:
    _T2I = T2I_ROOT
except NameError:
    _T2I = Path.cwd() if (Path.cwd() / 'data' / 'samples.jsonl').exists() else Path.cwd() / 'benchmark' / 't2i'

run_dir = _T2I / 'data' / GEN_RUN
bank = _T2I / 'data' / GEN_BANK_DIR / GEN_BANK
assert run_dir.exists(), f'批次目录不存在: {run_dir}'
assert bank.exists(), f'题库不存在: {bank}（先用 eval_synthesize.py 出题并合并）'

qs = {}
for line in bank.read_text(encoding='utf-8').splitlines():
    if line.strip():
        q = _json.loads(line)
        qs[q['qid']] = q
recs = [_json.loads(l) for l in (run_dir / 'responses_shard0.jsonl').open(encoding='utf-8') if l.strip()]

# —— 原图基线判分：每题把样本原图直接作为答案图，走 eval_score.py 同一判分链 ——
# 产物 run_dir/scores_orig/scores_<judge>.jsonl：⑤ 原图基线行 / ⑥ F_AUTHOR='原图' 自动消费。
# 按 qid 断点续跑（全齐零调用跳过；改过判官 prompt 须先删产物重跑）。
orig_scores = {}
if GEN_ORIG:
    import requests as _rq
    judges = GEN_ORIG_JUDGES or sorted(p.name[len('scores_'):-len('.jsonl')]
                                       for p in run_dir.glob('scores_*.jsonl'))
    assert judges, f'{run_dir} 下无 scores_*.jsonl（先跑生成图判分；原图基线须与生成图同判官方可比）'
    orig_dir = run_dir / 'scores_orig'
    orig_dir.mkdir(parents=True, exist_ok=True)
    want = {qid for qid, q in qs.items() if q.get('_sample_image')}
    resp_fp = orig_dir / 'responses_orig.jsonl'
    resp_fp.write_text(''.join(
        _json.dumps({'qid': qid, 'image': str(_T2I / 'data' / q['_sample_image'])}, ensure_ascii=False) + '\n'
        for qid, q in qs.items() if q.get('_sample_image')), encoding='utf-8')
    cons = _T2I / 'data' / _BT['probe_dir'] / _BT['probe_file']
    for j in judges:
        out_fp = orig_dir / f'scores_{j}.jsonl'
        have = ({_json.loads(l)['qid'] for l in out_fp.read_text(encoding='utf-8').splitlines() if l.strip()}
                if out_fp.exists() else set())
        if have >= want:
            print(f'[orig] {j}: 原图基线已全齐（{len(have)} 题），零调用跳过', flush=True)
        else:
            if GEN_ORIG_MODEL:
                model = GEN_ORIG_MODEL
            else:
                ids = [m['id'] for m in _rq.get(GEN_ORIG_ENDPOINT.split('/v1/')[0] + '/v1/models',
                                                 timeout=30).json()['data']
                       if m['id'].rsplit('/', 1)[-1] == j]
                assert ids, f'网关无判官 {j} 的路由，请手填 GEN_ORIG_MODEL'
                ids.sort(key=lambda i: (any(f'/{p}/' in f'/{i}/' for p in ('openrouter', 'galaxy')), len(i)))
                model = ids[0]
            print(f'[orig] {j}: 缺 {len(want - have)} 题 → {model} 补跑（原图作答案）', flush=True)
            _subp.run([_sys.executable, str(_T2I / 'eval_score.py'), 'score',
                       '--questions', str(bank), '--responses', str(resp_fp),
                       '--out', str(out_fp), '--endpoint', GEN_ORIG_ENDPOINT,
                       '--model', model, '--constraints', str(cons),
                       '--workers', str(GEN_ORIG_WORKERS)], check=True)
        orig_scores[j] = {}
        for l in out_fp.read_text(encoding='utf-8').splitlines():
            if l.strip():
                r = _json.loads(l)
                orig_scores[j][r['qid']] = r
    for j, m in orig_scores.items():
        ts = [r['total'] for r in m.values()]
        fs = [r['fidelity_score'] for r in m.values() if r.get('fidelity_score') is not None]
        print(f'[orig] {j}: 原图基线均分 {sum(ts) / len(ts):.1f} ｜ 保真线 {sum(fs) / len(fs):.1f}（n={len(ts)}）'
              if ts else f'[orig] {j}: 无记录')

def _short(m):
    return (m or '?').split('/')[-1]

# 按样本聚合
groups = {}
for r in recs:
    q = qs.get(r['qid'])
    if not q or not r.get('ok'):
        continue
    sid = q.get('_job_sample') or (q.get('_sample_image', '').split('/')[1][:4])
    groups.setdefault(sid, []).append((q, r))

def _img(p, h=260):
    if p and Path(p).exists():
        return f'<img src="{p}" loading="lazy" style="height:{h}px;width:{h}px;object-fit:cover">'
    return f'<div style="height:{h}px;width:{h}px;display:flex;align-items:center;justify-content:center;color:#c00;border:1px solid #eee">图缺失</div>'

def _orig_mean(qid):
    """原图基线总分（判官均值；无记录 None）——该题 checks 拿样本原图本身能得到的分。"""
    ov = [orig_scores[j][qid]['total'] for j in orig_scores if qid in orig_scores.get(j, {})]
    return sum(ov) / len(ov) if ov else None

sids = sorted(groups)
if GEN_SAMPLE:
    sids = [s for s in sids if s.startswith(GEN_SAMPLE)]
INV_COLOR = {'L2': '#b07020', 'L3': '#a33333'}
for sid in sids:
    items = groups[sid]
    sample_img = _T2I / 'data' / items[0][0]['_sample_image']
    label = items[0][0].get('_query_label', '')
    cols = [f"<div style='text-align:center'><div style='font-size:11px;color:#666'>样本图</div>{_img(sample_img)}</div>"]
    for q, r in sorted(items, key=lambda x: _short(x[0].get('_generator_model'))):
        inv = q.get('invocation', '?')
        ip = run_dir / r.get('image', '')
        om = _orig_mean(q['qid'])
        ob = (f'<span style="color:#6a5acd"> · 原图基线 <b>{om:.0f}</b></span>'
              if om is not None else '')
        cols.append(
            f"<div style='min-width:280px;max-width:320px'>"
            f"<div style='font-size:12px'><b>{_html.escape(_short(q.get('_generator_model')))}</b> "
            f"<span style='color:{INV_COLOR.get(inv, '#666')};font-weight:bold'>{_html.escape(inv)}</span> "
            f"<span style='color:#999'>{_html.escape(r['qid'][:34])}</span></div>"
            f"<div style='padding:4px;margin:4px 0;font-size:11.5px;line-height:1.5'>{_html.escape(q.get('gen_prompt', ''))}</div>"
            f"{_img(ip)}"
            f"<div style='font-size:11px;color:#888'>{r.get('seconds', '?')}s{ob}</div>"
            f"</div>")
    display(HTML(
        f"<div style='border:1px solid #999;padding:8px;margin:12px 0'>"
        f"<div style='font-size:13px;font-weight:bold'>{sid} · {_html.escape(label)} · {len(items)} 模型</div>"
        f"<div style='display:flex;gap:10px;align-items:flex-start;overflow-x:auto;margin-top:6px'>{''.join(cols)}</div>"
        f"</div>"))
print(f'共 {len(sids)} 个样本 / {sum(len(v) for v in groups.values())} 张生成图'
      + (f'；原图基线 {sum(len(m) for m in orig_scores.values())} 条（scores_orig/）' if orig_scores else ''))

In [ ]:
# ---- ⑤ 出题人质量对比（Bagel 得分视角：每样本图 × 各出题人 × 各判官）----
# 读 _BT['bagel_run'] 批 scores_<judge>.jsonl + 合并题库；判官均分 = 该题在各判官下的 overall 均值。
# 解读注意：分数低≠题差——可能是题更严/更深地暴露了 Bagel 短板；结合违反标记与判官分歧看。
# 表格内含题面（gen_prompt）与 Bagel 生成图缩略（点缩略图展开大图）。
import json as _json, html as _html
from pathlib import Path
from IPython.display import display, HTML

try:
    _T2I = T2I_ROOT
except NameError:
    _T2I = Path.cwd() if (Path.cwd() / 'data' / 'samples.jsonl').exists() else Path.cwd() / 'benchmark' / 't2i'

# ====== 批次参数（改版本在这里） ======
assert '_BT' in globals(), '先跑顶部「批次版本总开关」格'
BAGEL_RUN = _BT['bagel_run']               # ← 总开关
BANK_DIR  = _BT['bank_dir']                # ← 总开关
BANK_FILE = _BT['bagel_bank'] or _BT['bank_files'][0]      # ← 总开关
JUDGES    = []                            # 判官名单；空 = 自动发现该批 scores_<judge>.jsonl
# ====================================
RUN_DIR   = _T2I / 'data' / BAGEL_RUN
BANK      = _T2I / 'data' / BANK_DIR / BANK_FILE
if not JUDGES:
    JUDGES = sorted(p.name[len('scores_'):-len('.jsonl')]
                    for p in RUN_DIR.glob('scores_*.jsonl'))
    print(f'自动发现判官: {JUDGES}')
assert JUDGES, f'{RUN_DIR} 下无 scores_*.jsonl'

qs = {}
for line in BANK.read_text(encoding='utf-8').splitlines():
    if line.strip():
        q = _json.loads(line)
        qs[q['qid']] = q
scores = {}
for j in JUDGES:
    fp = RUN_DIR / f'scores_{j}.jsonl'
    if not fp.exists():
        print(f'[skip] 分数文件不存在: {fp}')
        continue
    scores[j] = {}
    for line in fp.read_text(encoding='utf-8').splitlines():
        if line.strip():
            r = _json.loads(line)
            scores[j][r['qid']] = r
# 原图基线：判官按同一套 checks 给样本原图（非 Bagel 生成图）的分
scores_orig = {}
for j in JUDGES:
    fp = RUN_DIR / 'scores_orig' / f'scores_{j}.jsonl'
    scores_orig[j] = {}
    if fp.exists():
        for line in fp.read_text(encoding='utf-8').splitlines():
            if line.strip():
                r = _json.loads(line)
                scores_orig[j][r['qid']] = r

def _author(q):
    return (q.get('_generator_model') or '?').split('/')[-1]

def _caps(r):
    s = ('N' if r.get('neg_critical_violated') else '') + ('A' if r.get('align_critical_violated') else '') + \
        ('Q' if r.get('quality_critical_violated') else '') + ('F' if r.get('fidelity_low') else '')
    return s or '·'

def _cell(r, ro=None):
    if r is None:
        return '<td style="color:#bbb">—</td>'
    caps = _caps(r)
    cap_col = '#c0392b' if caps != '·' else '#2a7a4b'
    orig = f'<span style="color:#6a5acd;font-size:10px">原{ro["total"]:.0f}</span>' if ro else ''
    return (f'<td style="text-align:center"><b>{r["total"]:.0f}</b> '
            f'<span style="color:{cap_col};font-size:10px">{caps}</span> {orig}<br>'
            f'<span style="color:#999;font-size:10px">保{r["fidelity_score"] if r["fidelity_score"] is not None else "—"} '
            f'对{r["alignment_score"] if r["alignment_score"] is not None else "—"}</span></td>')

def _thumb(p, small=56, big_h=420, big_w=460):
    """缩略图点击展开大图（details/summary，无 JS）"""
    if not p.exists():
        return '<span style="color:#bbb">图缺失</span>'
    return (f'<details><summary style="display:inline-block;cursor:zoom-in">'
            f'<img src="{p}" loading="lazy" style="height:{small}px;width:{small}px;object-fit:cover;vertical-align:middle"></summary>'
            f'<img src="{p}" loading="lazy" style="max-height:{big_h}px;max-width:{big_w}px;object-fit:contain"></details>')

samples = sorted({q.get('_job_sample') for q in qs.values()})
th = ''.join(f'<th style="padding:2px 8px">{j}</th>' for j in JUDGES) + '<th style="padding:2px 8px">判官均分</th>'

# 汇总排名表
rows_sum = []
for a in sorted({_author(q) for q in qs.values()}):
    aq = [qid for qid, q in qs.items() if _author(q) == a]
    per = []
    for j in JUDGES:
        vals = [scores[j][qid]['total'] for qid in aq if qid in scores[j]]
        per.append(sum(vals) / len(vals) if vals else None)
    ok = [v for v in per if v is not None]
    rows_sum.append((a, per, sum(ok) / len(ok) if ok else None))
rows_sum.sort(key=lambda x: -(x[2] or 0))
sum_html = ['<table border="1" style="border-collapse:collapse;font-size:12px"><tr><th>出题人（按均分排名）</th>' + th + '</tr>']
for a, per, m in rows_sum:
    tds = ''.join(f'<td style="text-align:center">{v:.1f}</td>' if v is not None else '<td>—</td>' for v in per)
    sum_html.append(f'<tr><td><b>{_html.escape(a)}</b></td>{tds}<td style="text-align:center"><b>{m:.1f}</b></td></tr>')
sum_html.append('</table>')
display(HTML('<div style="font-size:13px;font-weight:bold;margin:8px 0">出题人 × 判官 均分排名（其所有题的 overall 均值）</div>'
             + ''.join(sum_html)
             + '<div style="font-size:11px;color:#888;margin:4px 0">违反标记: N=defining负向 A=对齐0档 Q=质量缺陷0档 F=保真线&lt;40（v5.4.1 起纯权重聚合，标记仅诊断不钳分）；分数低可能是题更严而非题差；<span style="color:#6a5acd">原N</span>=判官用同一套 checks 评样本原图的基线总分</div>'))

# 逐样本卡：样本图 + 出题人 × 题面/生成图 × 判官明细
for sid in samples:
    qids = [qid for qid, q in qs.items() if q.get('_job_sample') == sid]
    label = qs[qids[0]].get('_query_label', '') if qids else ''
    ip = _T2I / 'data' / qs[qids[0]]['_sample_image']
    img = _thumb(ip, small=240, big_h=560, big_w=640)
    th_row = ('<th>出题人</th><th style="padding:2px 8px;min-width:380px">题目（题面 / Bagel 生成图）</th>' + th)
    tbl = [f'<table border="1" style="border-collapse:collapse;font-size:12px"><tr>{th_row}</tr>']
    for a in sorted({_author(qs[qid]) for qid in qids}):
        aq = [qid for qid in qids if _author(qs[qid]) == a]
        q = qs[aq[0]]
        tds = []
        for j in JUDGES:
            vals = [scores[j][qid] for qid in aq if qid in scores[j]]
            ovals = [scores_orig[j][qid] for qid in aq if qid in scores_orig.get(j, {})]
            tds.append(_cell(vals[0] if len(vals) == 1 else None,
                             ovals[0] if len(ovals) == 1 else None) if vals else '<td>—</td>')
        means = [sum(v['total'] for v in [scores[j][qid] for qid in aq if qid in scores[j]]) /
                 max(1, sum(1 for qid in aq if qid in scores[j])) for j in JUDGES]
        mean_all = sum(means) / len(means)
        qcell = (f'<div style="font-size:11px;min-width:360px;max-width:460px;line-height:1.45">{_html.escape(q.get("gen_prompt") or "")}</div>'
                 f'<div style="margin-top:4px">{_thumb(RUN_DIR / "imgs" / f"{aq[0]}.png")}'
                 f'<span style="font-size:10px;color:#999;margin-left:6px">{_html.escape(aq[0][:30])}'
                 f' · {q.get("invocation", "?")}</span></div>')
        tbl.append(f'<tr><td style="vertical-align:top"><b>{_html.escape(a)}</b></td>'
                   f'<td style="vertical-align:top">{qcell}</td>'
                   + ''.join(tds) + f'<td style="text-align:center"><b>{mean_all:.1f}</b></td></tr>')
    orig_tds = []
    for j in JUDGES:
        ov = [r['total'] for qid in qids for r in [scores_orig.get(j, {}).get(qid)] if r]
        orig_tds.append(f'<td style="text-align:center;color:#6a5acd"><b>{sum(ov)/len(ov):.1f}</b></td>' if ov else '<td>—</td>')
    om = [r['total'] for j in JUDGES for qid in qids for r in [scores_orig.get(j, {}).get(qid)] if r]
    tbl.append(f'<tr style="border-top:2px solid #6a5acd"><td style="vertical-align:top"><b style="color:#6a5acd">样本原图基线</b>'
               f'<div style="font-size:10px;color:#999">原图按各题 checks 判分均值</div></td>'
               + ''.join(orig_tds) + f'<td style="text-align:center"><b>{sum(om)/len(om):.1f}</b></td></tr>' if om else '')
    tbl.append('</table>')
    display(HTML(f'<div style="border:1px solid #999;padding:8px;margin:12px 0">'
                 f'<div style="font-size:13px;font-weight:bold">{sid} · {_html.escape(label)}'
                 f'<span style="font-size:11px;color:#888;font-weight:normal">　（样本原图缩略，点击展开；表内为题面与 Bagel 生成图缩略，点击展开）</span></div>'
                 f'<div style="margin:6px 0">{img}</div>'
                 + ''.join(tbl) + '</div>'))


In [ ]:
# ---- ⑥ 判分明细钻取：按样本/出题人/判官筛选，逐条 check 的得分与判官理由 ----
# 筛选参数（改这里）：
#   F_SAMPLE : 按样本筛选，如 '0008'；None = 不过滤
#   F_AUTHOR : 按出题人筛选（子串即可，可多个），如 'sol' 或 ['sol','glm']；
#              '原图' = 原图基线视角（④ 格 scores_orig/ 产物：样本原图直接作答案，题面/检查/判官全同，
#              主分展示原图记录、附 Bagel 生成图参照）；与出题人混选时按并集选卡；None = 不过滤
#   F_JUDGE  : 判官名（JUDGES 中之一）；'all' = 全部判官
F_SAMPLE = None
F_AUTHOR = '原图'
F_JUDGE  = 'gemini-3.7-flash'

import json as _json, html as _html, re as _re
from pathlib import Path
from IPython.display import display, HTML

try:
    _T2I = T2I_ROOT
except NameError:
    _T2I = Path.cwd() if (Path.cwd() / 'data' / 'samples.jsonl').exists() else Path.cwd() / 'benchmark' / 't2i'

# ====== 批次参数（改版本在这里；与出题人对比格一致） ======
assert '_BT' in globals(), '先跑顶部「批次版本总开关」格'
BAGEL_RUN = _BT['bagel_run']               # ← 总开关
BANK_DIR  = _BT['bank_dir']                # ← 总开关
BANK_FILE = _BT['bagel_bank'] or _BT['bank_files'][0]      # ← 总开关
JUDGES    = []                            # 空 = 自动发现 scores_*.jsonl
# ==========================================================
RUN_DIR = _T2I / 'data' / BAGEL_RUN
BANK    = _T2I / 'data' / BANK_DIR / BANK_FILE
if not JUDGES:
    JUDGES = sorted(p.name[len('scores_'):-len('.jsonl')]
                    for p in RUN_DIR.glob('scores_*.jsonl'))
    print(f'自动发现判官: {JUDGES}')
assert JUDGES, f'{RUN_DIR} 下无 scores_*.jsonl'

qs = {}
for line in BANK.read_text(encoding='utf-8').splitlines():
    if line.strip():
        q = _json.loads(line)
        qs[q['qid']] = q
scores = {}
for j in JUDGES:
    fp = RUN_DIR / f'scores_{j}.jsonl'
    if not fp.exists():
        print(f'[skip] 分数文件不存在: {fp}')
        continue
    scores[j] = {}
    for line in fp.read_text(encoding='utf-8').splitlines():
        if line.strip():
            r = _json.loads(line)
            scores[j][r['qid']] = r
# 原图基线：判官按同一套 checks 给样本原图打的分（④ 格 GEN_ORIG 产物）
scores_orig = {}
for j in JUDGES:
    fp = RUN_DIR / 'scores_orig' / f'scores_{j}.jsonl'
    scores_orig[j] = {}
    if fp.exists():
        for line in fp.read_text(encoding='utf-8').splitlines():
            if line.strip():
                r = _json.loads(line)
                scores_orig[j][r['qid']] = r

def _parse_judge_json(raw):
    """从 judge raw 输出尾部解析最终 JSON（含各细项 reason）。"""
    cand = None
    for m in _re.finditer(r'\{', raw or ''):
        try:
            obj = _json.loads(raw[m.start():])
        except Exception:
            continue
        if isinstance(obj, dict) and 'fidelity' in obj:
            cand = obj  # 取最后一个有效且含 fidelity 的
    return cand or {}

def _sc_badge(s):
    col = {0: '#c0392b', 1: '#b8860b', 2: '#2a7a4b'}.get(s, '#888')
    lab = {0: '0 未达', 1: '1 及格', 2: '2 满分'}.get(s, str(s) if s is not None else '—')
    return f'<span style="color:{col};font-weight:bold;white-space:nowrap">{lab}</span>'

def _author(q):
    return (q.get('_generator_model') or '?').split('/')[-1]

# 得分口径与 eval_score.py finalize_v53 对齐（v5.5.4）：
#   通用线 G = 一致/质量/美感三支柱等权均值；保真只作乘子进总分：
#   total = G × (FLOOR + (1-FLOOR) × F/100)，F 缺失退化为 G。
#   无封顶/熔断：*_violated / fidelity_low 仅诊断标记，不影响总分。
_FLOOR = 0.2

def _generic(r):
    g = r.get('generic_score')
    if g is not None:
        return g
    vals = [r.get(k) for k in ('alignment_score', 'quality_score', 'aesthetic_score') if r.get(k) is not None]
    return sum(vals) / len(vals) if vals else None

def _mult(r):
    f = r.get('fidelity_score')
    return 1.0 if f is None else _FLOOR + (1 - _FLOOR) * f / 100.0

def _diags(r):
    """诊断标记全称（仅诊断，不影响总分；避免缩写被误读）"""
    m = []
    if r.get('neg_critical_violated'):
        m.append('负向关键0档')
    if r.get('align_critical_violated'):
        m.append('一致项0档')
    if r.get('quality_critical_violated'):
        m.append('质量缺陷0档')
    if r.get('fidelity_low'):
        m.append(f'保真线低{r.get("fidelity_score")}<40')
    return m

def _check_rows(items, parsed_list, neg_idx=None):
    """items=题目 checks；parsed_list=judge 逐条输出 [{index,score,reason}]"""
    got = {c.get('index'): c for c in (parsed_list or [])}
    rows = []
    for i, c in enumerate(items, 1):
        g = got.get(i) or {}
        s, reason = g.get('score'), g.get('reason') or '（无理由）'
        is_neg = neg_idx and (i - 1) in neg_idx
        tag = (' <span style="background:#c0392b;color:#fff;border-radius:3px;padding:0 4px;font-size:10px">负·关键项</span>'
               if is_neg else '')
        rows.append(
            f'<tr style="vertical-align:top"><td style="padding:3px 6px;white-space:nowrap">F{i}{tag}</td>'
            f'<td style="padding:3px 6px;font-size:11.5px;min-width:280px">{_html.escape(c.get("check",""))}</td>'
            f'<td style="padding:3px 6px;text-align:center">{_sc_badge(s)}</td>'
            f'<td style="padding:3px 6px;font-size:11.5px">{_html.escape(str(reason))}</td></tr>')
    return ''.join(rows)

def _align_rows(items, parsed_align):
    """对齐明细：v5.5.3 起判官基于题面×图直判 Alignment 组 18 项全量（题面无该
    类规定记 N/A，出题端无 alignment_checks），parsed 为 facet->score 字典；
    旧批判分为 index 列表式，走按条渲染。"""
    if isinstance(parsed_align, dict):
        rows = []
        for fac, s in parsed_align.items():
            rows.append(
                f'<tr style="vertical-align:top"><td style="padding:3px 6px;white-space:nowrap">{_html.escape(str(fac))}</td>'
                f'<td style="padding:3px 6px;text-align:center">{_sc_badge(s)}</td></tr>')
        return ''.join(rows)
    got = {c.get('index'): c for c in (parsed_align or [])}
    rows = []
    for i, c in enumerate(items or [], 1):
        g = got.get(i) or {}
        s, reason = g.get('score'), g.get('reason') or '（无理由）'
        rows.append(
            f'<tr style="vertical-align:top"><td style="padding:3px 6px;white-space:nowrap">A{i}</td>'
            f'<td style="padding:3px 6px;font-size:11.5px;min-width:280px">{_html.escape(c.get("check",""))}</td>'
            f'<td style="padding:3px 6px;text-align:center">{_sc_badge(s)}</td>'
            f'<td style="padding:3px 6px;font-size:11.5px">{_html.escape(str(reason))}</td></tr>')
    return ''.join(rows)

_fa = [F_AUTHOR] if isinstance(F_AUTHOR, str) else list(F_AUTHOR) if F_AUTHOR else None
_orig_on = bool(_fa) and any(a == '原图' for a in _fa)   # 原图基线视角：主分读 scores_orig，附 Bagel 参照
_authors = [a for a in _fa or [] if a != '原图']
sel_judges = JUDGES if F_JUDGE in (None, 'all') else [F_JUDGE]
sel_qids = [qid for qid, q in sorted(qs.items())
            if (F_SAMPLE is None or q.get('_job_sample') == F_SAMPLE)
            and (not _authors or any(a.lower() in (q.get('_generator_model') or '').lower() for a in _authors))
            and (not _orig_on or any(qid in scores_orig.get(j, {}) for j in sel_judges))]
assert not sel_qids or all(j in scores for j in sel_judges), f'判官名应为 {JUDGES} 之一或 all'
if _orig_on and not any(scores_orig.get(j) for j in sel_judges):
    print(f'[提示] 所选判官无原图基线记录（{RUN_DIR / "scores_orig"}）——先跑 ④ 格 GEN_ORIG 判分')
if not sel_qids:
    display(HTML(f'<div style="color:#c00">无匹配题目（F_SAMPLE={F_SAMPLE!r}, F_AUTHOR={F_AUTHOR!r}）。'
                 f'可选样本: {sorted({q.get("_job_sample") for q in qs.values()})}</div>'))

_otag = '<span style="color:#6a5acd"> · 原图基线（样本原图作答案）</span>' if _orig_on else ''
for qid in sel_qids:
    q = qs[qid]
    ip = (_T2I / 'data' / q['_sample_image']) if _orig_on else (RUN_DIR / 'imgs' / f'{qid}.png')
    thumb = ('<details><summary style="display:inline-block;cursor:zoom-in">'
             f'<img src="{ip}" loading="lazy" style="height:72px;width:72px;object-fit:cover;vertical-align:middle"></summary>'
             f'<img src="{ip}" loading="lazy" style="max-height:460px;max-width:520px;object-fit:contain"></details>'
             f'<div style="margin-top:2px"><a href="{ip}" target="_blank" style="font-size:11px">打开高清原图 ↗</a></div>'
             if ip.exists() else '<span style="color:#bbb">生成图缺失</span>')
    body = [f'<div style="display:flex;gap:10px;align-items:flex-start;margin:4px 0">{thumb}'
            f'<div style="font-size:11.5px;max-width:520px"><b>题面</b>：{_html.escape(q.get("gen_prompt") or "")}<br>'
            f'<span style="color:#999">{_html.escape(qid)}</span></div></div>']
    for j in sel_judges:
        r = (scores_orig[j] if _orig_on else scores[j]).get(qid)
        if r is None:
            body.append(f'<div style="color:#c00;margin:6px 0">[{j}] 无判分记录'
                        + ('（原图基线）——先跑 ④ 格 GEN_ORIG 判分' if _orig_on else '') + '</div>')
            continue
        parsed = _parse_judge_json(r.get('raw'))
        diags = _diags(r)
        neg_idx = {int(x) for x in (r.get('neg_critical_idx') or [])}
        if _orig_on:
            rb = scores[j].get(qid)
            orig_html = (f' · <span style="color:#999;font-size:11px">参照 Bagel 生成图 <b>{rb["total"]:.1f}</b></span>'
                         if rb else '')
        else:
            ro = scores_orig.get(j, {}).get(qid)
            if ro:
                o_diags = _diags(ro)
                o_parsed = _parse_judge_json(ro.get('raw'))
                o_detail = ('<table border="1" style="border-collapse:collapse"><tr><th></th><th style="padding:2px 6px">原图 × 保真检查</th>'
                            '<th style="padding:2px 6px">档</th><th style="padding:2px 6px">判官理由</th></tr>'
                            + _check_rows(q.get('fidelity_checks') or [], o_parsed.get('fidelity'))
                            + '</table><table border="1" style="border-collapse:collapse;margin-top:4px"><tr><th></th><th style="padding:2px 6px">原图 × 一致性检查</th>'
                            '<th style="padding:2px 6px">档</th><th style="padding:2px 6px">判官理由</th></tr>'
                            + _align_rows(q.get('alignment_checks') or [], o_parsed.get('alignment')) + '</table>')
                orig_html = (f' · <span style="color:#6a5acd;font-size:11px">原图基线 <b>{ro["total"]:.1f}</b>'
                             f' <span style="color:#aaa">= G {_generic(ro):.1f} × 乘子 {_mult(ro):.2f}</span>'
                             + (f' 诊断：{_html.escape(" · ".join(o_diags))}' if o_diags else '')
                             + f'</span> <details style="display:inline"><summary style="display:inline;font-size:11px;color:#6a5acd;cursor:pointer">原图细项</summary>{o_detail}</details>')
            else:
                orig_html = ''
        _g = _generic(r)
        head = (f'<div style="font-size:12px;margin:8px 0 2px"><b>判官: {_html.escape(j)}</b> · '
                f'总分 <b>{r["total"]:.1f}</b>'
                + (f' <span style="color:#999;font-size:11px">= G {_g:.1f} × 保真乘子 {_mult(r):.2f}（地板 {_FLOOR}）</span>' if _g is not None else '')
                + (f' <span style="color:#c0392b;font-size:11px">诊断：{_html.escape(" · ".join(diags))}（不影响总分）</span>' if diags else ' <span style="color:#2a7a4b;font-size:11px">无诊断</span>')
                + f' · <span style="color:#999;font-size:11px">保真 {r["fidelity_score"]} / 一致 {r["alignment_score"]} / 质量 {r["quality_score"]} / 美感 {r["aesthetic_score"]}</span>'
                + orig_html + '</div>')
        # neg_critical_idx 在记录里即 0 基（negative_critical_idx 返回 enumerate 0 起索引）
        fid_tbl = ('<table border="1" style="border-collapse:collapse"><tr><th></th><th style="padding:2px 6px">保真检查</th>'
                   '<th style="padding:2px 6px">档</th><th style="padding:2px 6px">判官理由</th></tr>'
                   + _check_rows(q.get('fidelity_checks') or [], parsed.get('fidelity'), neg_idx) + '</table>')
        al_tbl = ('<table border="1" style="border-collapse:collapse;margin-top:4px"><tr><th></th><th style="padding:2px 6px">文本一致性检查</th>'
                  '<th style="padding:2px 6px">档</th><th style="padding:2px 6px">判官理由</th></tr>'
                  + _align_rows(q.get('alignment_checks') or [], parsed.get('alignment')) + '</table>')
        qa = []
        for name, obj in (('质量', parsed.get('quality') or {}), ('美感', parsed.get('aesthetics') or {})):
            parts = []
            for k, v in (obj.items() if isinstance(obj, dict) else []):
                s = v.get('score') if isinstance(v, dict) else v
                reason = v.get('reason') if isinstance(v, dict) else None
                parts.append(f'{_html.escape(str(k))} {_sc_badge(s)}'
                             + (f'<span style="color:#888;font-size:11px">：{_html.escape(str(reason))}</span>' if reason else ''))
            if parts:
                qa.append(f'<div style="font-size:11.5px;margin-top:4px"><b>{name}</b>：' + '； '.join(parts) + '</div>')
        body.append(head + fid_tbl + al_tbl + ''.join(qa))
    display(HTML(f'<div style="border:1px solid #999;padding:8px;margin:12px 0">'
                 f'<div style="font-size:13px;font-weight:bold">出题人: {_html.escape(_author(q))}{_otag}'
                 f'<span style="color:#888;font-weight:normal"> · 样本 {q.get("_job_sample")} · {_html.escape(q.get("_query_label") or "")} · {q.get("invocation","?")}</span>'
                 f'<span style="color:#999;font-size:11px;font-weight:normal">　[{_html.escape(qid)}]</span></div>'
                 + ''.join(body) + '</div>'))

In [ ]:
# ---- ⑦ 横向模型对比（原始不泄漏题面 × N 个生成模型：得分矩阵 + 逐题图卡） ----
# 读 eval_bagel_v55 / eval_bagel_v55_leak / eval_models_v55 的判分与图；
# 判分题面统一 = eval_bagel_v55/questions.jsonl 原始题（判官看不到泄漏），与 ⑤⑥ 同口径。
# 参数（改这里）：
#   F_SCORE : 外部模型分数文件过滤子串（对 scores_*.jsonl），None = 全部
F_SCORE = None

import json as _json, html as _html
from pathlib import Path
from IPython.display import display, HTML

try:
    _T2I = T2I_ROOT
except NameError:
    _T2I = Path.cwd() if (Path.cwd() / 'data' / 'samples.jsonl').exists() else Path.cwd() / 'benchmark' / 't2i'
D = _T2I / 'data'

qs = {}
for line in (D / 'eval_bagel_v55' / 'questions.jsonl').read_text(encoding='utf-8').splitlines():
    if line.strip():
        q = _json.loads(line)
        qs[q['qid']] = q
qids = sorted(qs)

COLS = [('Bagel 基线',   D / 'eval_bagel_v55'      / 'scores_gemini-3.7-flash.jsonl', D / 'eval_bagel_v55'      / 'imgs'),
        ('Bagel 泄漏版', D / 'eval_bagel_v55_leak' / 'scores_gemini-3.7-flash.jsonl', D / 'eval_bagel_v55_leak' / 'imgs')]
_md = D / 'eval_models_v55'
if _md.exists():
    for _sf in sorted(_md.glob('scores_*.jsonl')):
        if F_SCORE and F_SCORE not in _sf.name:
            continue
        _sn = _sf.name[len('scores_'):-len('.jsonl')]
        COLS.append((_sn, _sf, _md / 'imgs' / _sn))

cols = []
for label, sf, imgdir in COLS:
    sc = {}
    sf = Path(sf)
    if sf.exists():
        for line in sf.read_text(encoding='utf-8').splitlines():
            if line.strip():
                r = _json.loads(line)
                sc[r['qid']] = r
    cols.append((label, sc, Path(imgdir)))

def _sc_badge(s):
    try:
        t, v = int(s), float(s)
        col = {0: '#c0392b', 1: '#b8860b', 2: '#2a7a4b'}.get(t, '#888')
        lab = {0: '0未达', 1: '1及格', 2: '2满分'}.get(t, str(s))
        return f'<span style="color:{col};font-weight:bold;white-space:nowrap">{lab}</span>'
    except Exception:
        return str(s) if s is not None else '-'

def _tot_bg(v):
    if v is None:
        return ''
    return 'background:' + ('#e8f5e9' if v >= 40 else '#fff8e1' if v >= 20 else '#fdecea') + ';'

# ====== A. 得分矩阵：行=题目，列=模型（总分 + 保真档/一致/质量/美感；末行均值） ======
rows = ['<tr><th style="padding:3px 6px">qid</th>' + ''.join(f'<th style="padding:3px 6px">{_html.escape(str(l))}</th>' for l, _, _ in cols) + '</tr>']
for q in qids:
    tds = [f'<td style="padding:2px 6px;font-size:11px;white-space:nowrap">{_html.escape(q)}<br>'
           f'<span style="color:#999;font-size:10px">{_html.escape(qs[q].get("_query_label", ""))}</span></td>']
    for _l, sc, _imgd in cols:
        r = sc.get(q)
        if not r:
            tds.append('<td style="padding:2px 6px;color:#bbb;text-align:center">-</td>')
            continue
        mark = ''
        if r.get('neg_critical_violated') or r.get('align_critical_violated') or r.get('quality_critical_violated'):
            mark = ' <span style="color:#c0392b" title="存在关键项0档">!</span>'
        tds.append(f'<td style="padding:2px 6px;text-align:center;{_tot_bg(r.get("total"))}">'
                   f'<b>{r["total"]:.1f}</b>{mark}<br><span style="font-size:10px;color:#888">'
                   f'保 {_sc_badge(r.get("fidelity_score"))} / 齐 {r.get("alignment_score")} / 质 {r.get("quality_score")} / 美 {r.get("aesthetic_score")}</span></td>')
    rows.append('<tr>' + ''.join(tds) + '</tr>')
means = []
for _l, sc, _imgd in cols:
    vals = [sc[q]['total'] for q in qids if q in sc]
    means.append(sum(vals) / len(vals) if vals else None)
rows.append('<tr style="border-top:2px solid #666"><th style="padding:3px 6px">均值</th>' +
            ''.join(f'<td style="padding:3px 6px;text-align:center;font-weight:bold;{_tot_bg(m)}">{m:.1f}</td>' if m is not None else '<td style="color:#bbb;text-align:center">-</td>' for m in means) + '</tr>')
display(HTML('<div style="overflow-x:auto"><table border="1" style="border-collapse:collapse">' + ''.join(rows) + '</table></div>'
             '<div style="font-size:11px;color:#888;margin-top:4px">total = G × (0.2 + 0.8×F/100)（QIB φ 映射 {0,60,100}）；保真列显示保真线得分，括号内为档。'
             '判分题面 = 原题库题面（不含泄漏），所有列同口径可比。</div>'))

# ====== B. 逐题图卡：同一题各模型生成图并排（点击放大；题面可展开） ======
for q in qids:
    meta = qs[q]
    cards = []
    for label, sc, imgdir in cols:
        ip = imgdir / f'{q}.png'
        r = sc.get(q)
        if r is not None:
            cap = f'{_html.escape(str(label))}<br>总分 <b>{r["total"]:.1f}</b> / 保真 {r.get("fidelity_score")}'
            if r.get('fidelity_low'):
                cap += ' <span style="color:#c0392b">低</span>'
        else:
            cap = f'{_html.escape(str(label))}<br><span style="color:#bbb">无判分</span>'
        if ip.exists():
            img = (f'<details style="display:inline-block;vertical-align:top;margin:0 6px"><summary style="cursor:zoom-in">'
                   f'<img src="{ip}" loading="lazy" style="height:110px;width:110px;object-fit:cover;vertical-align:middle"></summary>'
                   f'<img src="{ip}" loading="lazy" style="max-height:480px;max-width:480px;object-fit:contain"></details>')
        else:
            img = '<div style="display:inline-block;vertical-align:top;margin:0 6px;width:110px;height:110px;background:#f0f0f0;line-height:110px;text-align:center;color:#aaa">缺图</div>'
        cards.append(f'<div style="display:inline-block;text-align:center;vertical-align:top">{img}<div style="font-size:11px;max-width:150px">{cap}</div></div>')
    display(HTML(
        f'<div style="border:1px solid #999;border-radius:6px;padding:8px;margin:10px 0">'
        f'<div style="font-size:12.5px;font-weight:bold">{_html.escape(q)} <span style="color:#666;font-weight:normal">· '
        f'{_html.escape(meta.get("_query_label", ""))} · {meta.get("invocation", "")}</span> '
        f'<details style="display:inline"><summary style="display:inline;font-size:11px;color:#2a7a4b;cursor:pointer">题面(判分依据，不含泄漏)</summary>'
        f'<div style="font-size:11.5px;max-width:900px;margin:4px 0;white-space:pre-wrap">{_html.escape(meta.get("gen_prompt") or "")}</div></details></div>'
        f'<div style="margin-top:6px">' + ''.join(cards) + '</div></div>'))


## 影子批审阅（视觉知识检测线：镜头 + 兜底；协议验证批，不入题库）

> 自动选批：`eval_v57_shadow`（v5.7 线索制）优先，缺失回退 `eval_v56_shadow`。只读产物，不依赖前面格子。
> 出题为影子批（验证协议用，未入题库）；正式批复用时把路径常量换成正式批目录即可。


In [ ]:
# ---- 影子批审阅（协议验证批，不入题库；v5.7 线索制优先，自动回退 v5.6；只读产物） ----
import json as _json
import html as _html
from pathlib import Path
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

try:
    T2I_ROOT                                            # 加载格已跑则复用
except NameError:
    T2I_ROOT = Path.cwd() if (Path.cwd() / 'data' / 'samples.jsonl').exists() else Path.cwd() / 'benchmark' / 't2i'

_DATA = T2I_ROOT / 'data'
# 自动选批：v5.7 影子批存在就用它，否则回退 v5.6
_V57_DIR, _V56_DIR = _DATA / 'eval_v57_shadow', _DATA / 'eval_v56_shadow'
if (_V57_DIR / 'scores_t2i.jsonl').exists():
    _SHADOW_TAG, _SCORES_F, _RESP_F = 'v5.7', _V57_DIR / 'scores_t2i.jsonl', _V57_DIR / 'responses_gemini-3.1-flash-lite-image.jsonl'
    _QUESTIONS_F = _DATA / 'synth_gen_v57_shadow' / 'questions_v57_gpt-5.6-sol.jsonl'
elif (_V56_DIR / 'scores_t2i.jsonl').exists():
    _SHADOW_TAG, _SCORES_F, _RESP_F = 'v5.6', _V56_DIR / 'scores_t2i.jsonl', _V56_DIR / 'responses_gemini-3.1-flash-lite-image.jsonl'
    _QUESTIONS_F = _DATA / 'synth_gen_v56_shadow' / 'questions_v56_gpt-5.6-sol.jsonl'
else:
    raise FileNotFoundError('影子批判分产物不存在（先跑 eval_score.py score）')

_LENS = {'K1': '形态与构成', 'K2': '表面与图样', 'K3': '符号与文字',
         'K4': '数量与尺度', 'K5': '状态与规律'}
_SCORE_COL = {0: '#c0392b', 1: '#b07020', 2: '#2a7a4b',
              0.0: '#c0392b', 60.0: '#b07020', 100.0: '#2a7a4b'}

def _load(fp):
    return [_json.loads(l) for l in fp.read_text(encoding='utf-8').splitlines() if l.strip()]

_qs = _load(_QUESTIONS_F)
_resp = {r['qid']: r for r in _load(_RESP_F)}
_scores = {r['qid']: r for r in _load(_SCORES_F)}
_IS_V57 = any(r.get('schema') == 'v5.7' for r in _scores.values())
print(f'影子批 {_SHADOW_TAG}（schema {"v5.7 线索制" if _IS_V57 else "v5.6 逐条制"}）: '
      f'{len(_qs)} 题 / 判分 {len(_scores)} 题 / 判官 2.5-pro / 生图 gemini-3.1-flash-lite-image')

# ---- 1. 汇总表 ----
_rows = []
for q in _qs:
    r = _scores.get(q['qid'])
    if r is None:
        continue
    dist = Counter(c.get('lens', '?') for c in q.get('fidelity_checks', []))
    row = {'qid': q['qid'], '概念': q.get('_query_label', '?'), '调用': q.get('invocation'),
           '线索数': len(q.get('fidelity_checks', [])),
           '镜头分布': ' '.join(f'{k}×{dist[k]}' for k in sorted(dist)),
           'K线': r['fidelity_score'], '总分': r['total'],
           '兜底': len(r.get('knowledge_fallback') or [])}
    if _IS_V57:
        row['K维度分'] = ' '.join(f'{k}:{int(v)}' for k, v in sorted((r.get('knowledge_dims') or {}).items()))
    else:
        row['对齐'], row['质量'], row['美感'] = r['alignment_score'], r['quality_score'], r['aesthetic_score']
    _rows.append(row)
_v57_df = pd.DataFrame(_rows).sort_values('总分')
with pd.option_context('display.max_rows', None):
    display(_v57_df)

# ---- 2. 镜头切片 ----
_lens_scores = {}
for q in _qs:
    r = _scores.get(q['qid'])
    if r is None:
        continue
    if _IS_V57:
        for l, v in (r.get('knowledge_dims') or {}).items():
            _lens_scores.setdefault(l, []).append(v)          # 维度级 0/60/100
    else:
        fs = r.get('fidelity_scores') or []
        for i, c in enumerate(q.get('fidelity_checks', [])):
            if i < len(fs) and fs[i] in (0, 1, 2):
                _lens_scores.setdefault(c.get('lens', '?'), []).append(fs[i] * 50.0)
fig, ax = plt.subplots(1, 1, figsize=(7, 3.4))
_lbl, _avg, _n = [], [], []
for l in sorted(_lens_scores):
    v = _lens_scores[l]
    _lbl.append(f'{l} {_LENS.get(l, "")}')
    _avg.append(sum(v) / len(v))
    _n.append(len(v))
ax.barh(_lbl[::-1], _avg[::-1], color='#8ab')
for i, (a, n) in enumerate(zip(_avg[::-1], _n[::-1])):
    ax.text(a + 2, i, f'{a:.0f} (n={n})', va='center', fontsize=8)
ax.set_xlim(0, 112)
ax.set_title(f'镜头 × {"维度分" if _IS_V57 else "检查分"}（0/60/100 制）')
plt.tight_layout(); plt.show()

# ---- 3. 兜底命中（判官在 10 条之外发现的知识错误；不进分，供挖掘反哺） ----
_fb_rows = [{'qid': qid, '错误点': f.get('issue', ''), '应然事实': f.get('should_be', '')}
            for qid, r in _scores.items() for f in (r.get('knowledge_fallback') or [])]
if _fb_rows:
    with pd.option_context('display.max_colwidth', 60, 'display.max_rows', None):
        display(pd.DataFrame(_fb_rows))
else:
    print('兜底无命中')

# ---- 4. 逐题图卡（生成图 + 题面 + 线索/判分） ----
def _esc(x):
    return _html.escape(str(x))

def _card(q):
    r = _scores.get(q['qid'])
    resp = _resp.get(q['qid']) or {}
    img_p = None
    if resp.get('ok') and resp.get('image'):
        cand = _SCORES_F.parent / resp['image']
        if cand.exists():
            img_p = cand
    img_html = (f'<img src="{img_p}" loading="lazy" style="max-height:320px;max-width:420px;object-fit:contain">'
                if img_p else '<div style="color:#c00">生成图缺失</div>')
    if _IS_V57:
        dims = (r or {}).get('knowledge_dims') or {}
        reasons = (r or {}).get('knowledge_reasons') or {}
        by_lens = {}
        for c in q.get('fidelity_checks', []):
            by_lens.setdefault(c.get('lens', '?'), []).append(c)
        sec = []
        for lk in ('K1', 'K2', 'K3', 'K4', 'K5'):
            if lk not in by_lens:
                continue
            v = dims.get(lk)
            chip = (f'<span style="background:{_SCORE_COL.get(v, "#666")};color:#fff;border-radius:3px;'
                    f'padding:0 5px;font-size:11px">维度判 {int(v)}</span>' if v is not None else '')
            items = ''.join(f'<div style="margin:2px 0 2px 14px">· {_esc(c.get("check", ""))}'
                            + (f' <span style="color:#888;font-size:11px">（可见: {_esc(c.get("visibility_requirement", ""))}）</span>'
                               if c.get('visibility_requirement') else '') + '</div>'
                            for c in by_lens[lk])
            reason = reasons.get(lk) or ''
            sec.append(f'<div style="margin:6px 0"><b>{lk} {_LENS.get(lk, "")}</b> {chip}{items}'
                       + (f'<div style="color:#444;margin-left:14px;font-size:12px">判官: {_esc(reason)}</div>' if reason else '')
                       + '</div>')
        checks_html = ''.join(sec)
        title = f'[世界知识线 · 按镜头维度判分]（{len(q.get("fidelity_checks", []))} 条线索 / {len(dims)} 维）'
    else:
        fs = (r or {}).get('fidelity_scores') or []
        reasons_map = (r or {}).get('fidelity_reasons') or {}
        items = []
        for i, c in enumerate(q.get('fidelity_checks', []), 1):
            sc = fs[i - 1] if i - 1 < len(fs) else None
            chip = (f'<span style="background:{_SCORE_COL.get(sc, "#666")};color:#fff;border-radius:3px;'
                    f'padding:0 5px;font-size:11px">判 {sc}</span>' if sc in (0, 1, 2) else '')
            reason = reasons_map.get(str(i)) or ''
            items.append(f'<div style="margin:6px 0"><b>{i}. {_esc(c.get("check", ""))}</b> '
                         f'<span style="background:#25648a;color:#fff;border-radius:3px;padding:0 4px;font-size:11px">{c.get("lens", "?")}</span> {chip}'
                         + (f'<div style="color:#444;margin-left:14px;font-size:12px">判官: {_esc(reason)}</div>' if reason else '')
                         + '</div>')
        checks_html = ''.join(items)
        title = f'[世界知识线 · 逐条判分]（{len(q.get("fidelity_checks", []))} 条）'
    # 通用三支柱逐项明细（对齐 18 / 质量 7 / 美感 3，QIB 四档；证据直显）
    _PILLAR_COL = {0: '#c0392b', 1: '#b07020', 2: '#2a7a4b'}
    SHOW_NA_REASON = True          # N/A 项的依据也列出
    pillar_rows = []
    for pname, key, rkey in (('对齐', 'alignment_scores', 'alignment_reasons'),
                             ('质量', 'quality_scores', 'quality_reasons'),
                             ('美感', 'aesthetic_scores', 'aesthetic_reasons')):
        d = (r or {}).get(key) or {}
        rs = (r or {}).get(rkey) or {}
        if not d:
            continue
        chips, detail = [], []
        for t, v in d.items():
            why = (rs.get(t) or '').strip()
            if v in ('N/A', None):
                chips.append(f'<span style="color:#999;font-size:11px">{t}:N/A</span>')
                if SHOW_NA_REASON and why:
                    detail.append(f'<div style="color:#999;margin:1px 0 1px 14px;font-size:11px">{t} [N/A]：{_esc(why)}</div>')
            else:
                chips.append(f'<span style="background:{_PILLAR_COL.get(v, "#666")};color:#fff;border-radius:3px;'
                             f'padding:0 4px;font-size:11px">{t}:{v}</span>')
                if why:
                    col = '#c0392b' if v == 0 else '#444'
                    detail.append(f'<div style="color:{col};margin:1px 0 1px 14px;font-size:11px">{t} [{v}]：{_esc(why)}</div>')
        n_na = sum(1 for v in d.values() if v in ('N/A', None))
        pillar_rows.append(f'<div style="margin:6px 0"><b>{pname}</b>'
                           f'（{len(d) - n_na} 判 / {n_na} N/A） {"　".join(chips)}</div>'
                           + ''.join(detail))
    pillar_html = ('<div style="margin-top:8px;border-top:1px dashed #ccc;padding-top:6px;font-size:12px">'
                   '<b>[通用三支柱明细]</b>' + ''.join(pillar_rows) + '</div>') if pillar_rows else ''

    fb = (r or {}).get('knowledge_fallback') or []
    fb_html = ''.join(f'<div style="color:#a33333;margin:3px 0">· {_esc(f.get("issue", ""))}（应然: {_esc(f.get("should_be", ""))[:80]}…）</div>'
                      for f in fb)
    score_line = ''
    if r:
        if _IS_V57:
            score_line = (f'<div style="font-size:12px;margin-top:4px"><b>判分</b>: K线 {r["fidelity_score"]} · '
                          f'对齐 {r["alignment_score"]} · 质量 {r["quality_score"]} · 美感 {r["aesthetic_score"]} · '
                          f'<b>总分 {r["total"]}</b>（total=G×K/100）</div>')
        else:
            score_line = (f'<div style="font-size:12px;margin-top:4px"><b>判分</b>: K线 {r["fidelity_score"]} · '
                          f'对齐 {r["alignment_score"]} · 质量 {r["quality_score"]} · 美感 {r["aesthetic_score"]} · '
                          f'<b>总分 {r["total"]}</b></div>')
    display(HTML(
        f'<div style="border:1px solid #ddd;border-left:4px solid #25648a;padding:8px 12px;margin:10px 0">'
        f'<div style="font-size:13px"><b>{_esc(q.get("qid", "?"))}</b> · {_esc(q.get("_query_label", "?"))} · '
        f'{_esc(q.get("invocation", "?"))} · schema {_esc((r or {}).get("schema", "?"))}</div>'
        f'<div style="display:flex;gap:12px;margin:6px 0;align-items:flex-start">{img_html}'
        f'<div style="min-width:0;flex:1"><div style="padding:4px;font-size:13px">{_esc(q.get("gen_prompt", ""))}</div>{score_line}</div></div>'
        f'<div style="margin-top:6px;font-size:12px"><b>{title}</b>{checks_html}</div>{pillar_html}'
        + (f'<div style="margin-top:6px;font-size:12px"><b>[兜底命中]</b>{fb_html}</div>' if fb_html else '')
        + '</div>'))

for _q in sorted(_qs, key=lambda x: x.get('qid', '')):
    _card(_q)
